# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mishellscripts/flyrank/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Given limited editorial time, which declining pages
should be reviewed first? Can we predict recovery from a trained model and would it beat a hand-built rule?
A generated score consisting of decline severity and recovery potential is a mathematical approach that supports the decision-making process of content editors when manually checking content items.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

The data used is the FlyRank internship warehouse dataset, specifically `fact_content_daily_performance` (daily impressions/clicks/position) and
`dim_content` (static content attributes). A cutoff T = 2026-02-01 was chosen by
scanning every candidate month for the earliest date with maximum # of clients with both
sufficient feature history and label runway. This resulted in 39 eligible clients.

Features excluded were:
- `trend_pct`/`trend_direction` as model features that are used only to define the decline gate itself
- `is_deleted`, `is_published` are current-snapshot
fields with risk of reflecting post-T state
- `last_optimized_date` used as a feature would risk leakage specifically because "was this page optimized" and "did this page recover" are plausibly the same event
- `client_hash_id`/`content_hash_id` used only for joins and
GroupKFold groupings

The data is anonymized with no client names, URLs, or private queries anywhere.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

For recovery prediction, both logistic regression and random forest were used. Both techniques were used with 5-fold cross valiation to compare model quality based on AUC and precision. With a small dataset, it was important to use cross-validation to lower overfitting risk to memorized client training data.

Features were deliberately chosen - Every feature listed in section 2 were not included. Furthermore, the cross-validated folds did not contain the same repeat clients. The recovery label windows and feature windows were strictly separated by the cutoff T and never overlapped. The label `recovered_by_T1` represents an increase in impressions from one 30d period to the next after T. This value was compared to the prediction model responses for precision.

After checking for multicollinearity, some correlated features were discovered. In particular, a cluster of model_used, char_count, char_count_imputed, and ai_generated correlated
with each other, dominating the model's coefficients. PCA compressed the confound cluster into orthogonal components before
model fitting, resolving the correlation directly in the feature space. Elastic Net resolved the same correlation during model
fitting rather than beforehand. Both PCA and Elastic Net improved the model slightly. Elastic Net was selected as the final feature selection method because it handled correlated predictors
directly with similar performance without needing PCA's feature-blending complexity. For the final feature set, the random forest max depth was tuned via GroupKFold-aware grid search.

For scoring, the baseline and final model share the same
formula: impact_at_risk * unlikeliness-to-recover. The baseline used avg_position as an
untested proxy for recovery likelihood. The final model replaces that proxy with a probability of recovery, a cross-validated
prediction from real historical outcomes.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

The hand rule baseline was built and evaluated on the starter
CSV (a single 90-day snapshot, no forward-looking outcome available). The CSV also has no
recovered_by_T1 label, so the baseline was never validated against
recovery. What is comparable: the baseline's own review
found real weaknesses on its own terms — 4 of its top 20 picks
were labeled low_decline yet outranked high_decline picks, since its hard
cutoff doesn't match the continuous score, and 2 top picks had fewer
than 10 clicks despite ranking high on impressions alone. Both weaknesses
are exactly what a trained, continuous p_recovery was meant to fix.

Instead, we can use a base rate measuring fraction of pages that would count as "correct" from
picking at random, given each fold's own natural mix of recovered vs.
stayed-broken pages. Lift subtracts out that base-rate effect: an average lift
of 0.29–0.31 means the model's top picks beat random guessing by roughly
29–31 percentage points, on average, after accounting for how easy or hard
each fold's base rate already made the task.

| | Full features | Elastic Net (final) |
|---|:---:|:---:|
| AUC (random forest) | 0.684 | 0.708 |
| Precision@20 | 0.890 | 0.890 |
| Precision@50 | 0.892 | 0.912000 |
| Precision@20 lift over base rate | 0.292 | 0.292 |
| Precision@50 lift over base rate | 0.294 | 0.314 |

## 5. Limitations

*What this work cannot claim.*

No causal claim can be made that reviewing a flagged
page causes recovery. The model was trained to identify pages worth a human review, not to make the review decision itself. Never auto-remove, auto-merge, or auto-deprioritize a page based on this queue alone. Results can be inaccurate so an editor should review not only the reason code, but also the continuous priority score value, impact_at_risk, and p_recovery and use their best judgment.

Model was built on only 39 eligible clients. Scores for very new or
thin-history clients should be treated as lower confidence. The model should be retrained when there is more data available.
This queue answers "which known problems deserve attention first," not
"which healthy pages are about to decline". That would be a different, complementary
model that could be added in to increase the client pool.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.